# Block Ciphers

In [ ]:
from random import randbytes, randrange
from typing import Literal
from itertools import product, zip_longest
from typing import Iterator

import numpy as np
from scipy.linalg import circulant
from numpy.typing import NDArray

from aes import AES
from blockcipher import BlockCipher


In [ ]:
# %matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.image import imread

Unlike **stream ciphers**, which encrypt data one bit at a time, **block ciphers** encrypt data in fixed-size blocks. For example, **AES** encrypts blocks of **128 bits**.

Since a block cipher can only encrypt a single block under a fixed key, several **modes of operation** have been designed. These modes allow block ciphers to be used repeatedly and securely on longer messages.

Moreover, block ciphers can also be used as building blocks in other cryptographic protocols, such as:
- universal hash functions
- pseudo-random number generators

![Caption](images/block_ciphers.png)

## Advanced Encryption Standard (AES)



**AES** is a specification for **symmetric-key encryption** established by **NIST** in **2001** and later adopted by the **U.S. government**.

The AES standard includes three block ciphers derived from the Rijndael cipher family.  
Each cipher has a fixed **128-bit block size**, but supports different key sizes:

| Key length | Number of rounds |
|----------:|-----------------:|
| 128 bits  | 10 rounds |
| 192 bits  | 12 rounds |
| 256 bits  | 14 rounds |

AES is composed of two main parts:

- a **key expansion block**
- a **data path**

The data path can be viewed as an **iterated block cipher**, where each iteration is called a **round**.

The number of rounds depends on the key length.

**References**: [FIPS PUB 197](https://csrc.nist.gov/pubs/fips/197/final), *Advanced Encryption Standard (AES)*, National Institute of Standards and Technology, U.S. Department of Commerce, November 2001.


## The Rijndael Block Cipher

Cryptographic algorithm designed by Joan Daemen and Vincent Rijmen (2) that became the basis of AES (AES is the standardized 128-bit-block version of Rijndael) .

The **round transformation** of Rijndael/AES is composed of three distinct invertible transformations, called **layers**.

In these layers, every bit of the state is treated in a similar way.

The three layers are:
1) <span style="color:#E57373"><strong>Non-linear layer</strong></span>: provides **confusion**
2) <span style="color:#2E8B57"><strong>Linear mixing layer</strong></span>: provides **diffusion**
3) <span style="color:#1E90FF"><strong>Key addition layer</strong></span>: inserts the key into the datapath

**References**: 2. Joan Daemen and Vincent Rijmen, [*The Design of Rijndael: AES — The Advanced Encryption Standard*](https://link.springer.com/book/10.1007/978-3-662-04722-4), Springer-Verlag, 2002.

![Caption](images/Rijndael_Block_Cipher.png)

The **internal state** of AES is a sequence of **128 bits** (**16 bytes**), represented as a **4×4 matrix of bytes**.

In [ ]:
block_size = 128 // 8
state = randbytes(block_size)

In [ ]:
def print_state(state):
    print(''.join([f'{byte:02x}' for j, byte in enumerate(state)]))

def show_state(state, ax=None, title=None):
    mat = np.array(list(state), dtype=np.uint8).reshape(-1, 4).T
    shape = mat.shape

    if ax is None:
        fig, ax = plt.subplots(figsize=(shape[0]/2, shape[1]/2))

    ax.matshow(mat, vmin=0, vmax=255, cmap='gray')
    ax.set(xticks=[], yticks=[])
    if title is not None:
        ax.set(title=title)

    for i, j in product(range(shape[0]), range(shape[1])):
        color = 'w' if mat[i, j] < 128 else 'k'
        text = ax.text(j, i, f'{mat[i, j]:02x}',
                       ha="center", va="center", color=color)

In [ ]:
show_state(state)
plt.show()

### <span style="color:#1E90FF"><strong>AddRoundKey</strong></span>

Internally, Rijndael represents the 128-bit state as a 4 × 4 matrix of bytes.

The **AddRoundKey** operation consists of a bitwise XOR between the current state and the round key derived from the cipher key:

$
Bj = Aj \oplus Kj
$

This step is the **key addition layer**, because it inserts the secret key material into the encryption process.

|  | Hex | Binary |
|:----------|:---:|:--------:|
| State byte | `53 53 53 53 53 53 53 53 53 53 53 53 53 53 53 53` | `01010011`$\times 16$ |
| Round key byte | `2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f 2f` | `00101111`$\times 16$ |
| Result byte | `7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c 7c` | `01111100`$\times 16$ |

In [ ]:
def add_round_key(state: bytes, round_key: bytes) -> bytes:
    return bytes(s ^ k for s, k in zip(state, round_key))

Tutorial **Type hinting**: is a way of showing the expected data types of function inputs and outputs.

For example:

- `state: bytes` means `state` should be a `bytes` value.
- `round_key: bytes` means `round_key` should be a `bytes` value.
- `-> bytes` means the function should return a `bytes` value.

Type hinting is useful because it makes code easier to read, helps find mistakes earlier, and improves editor support such as autocomplete.

In [ ]:
round_key = randbytes(block_size)
state_input = randbytes(block_size)
state_output = add_round_key(state_input, round_key)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6, 2))
show_state(round_key, ax=ax1, title='key')
show_state(state_input, ax=ax2, title='input')
show_state(state_output, ax=ax3, title='output')
plt.show()

### <span style="color:#E57373"><strong>Byte substitution</strong></span>:
is a non-linear byte substitution that operates independently on each byte of the State using a substitution table (S-box).



![Caption](images/byte_substitution.png)

In [ ]:
SBOX = bytes.fromhex(
        # 0 1 2 3 4 5 6 7 8 9 a b c d e f
        '637c777bf26b6fc53001672bfed7ab76' # 0
        'ca82c97dfa5947f0add4a2af9ca472c0' # 1
        'b7fd9326363ff7cc34a5e5f171d83115' # 2
        '04c723c31896059a071280e2eb27b275' # 3
        '09832c1a1b6e5aa0523bd6b329e32f84' # 4
        '53d100ed20fcb15b6acbbe394a4c58cf' # 5
        'd0efaafb434d338545f9027f503c9fa8' # 6
        '51a3408f929d38f5bcb6da2110fff3d2' # 7
        'cd0c13ec5f974417c4a77e3d645d1973' # 8
        '60814fdc222a908846eeb814de5e0bdb' # 9
        'e0323a0a4906245cc2d3ac629195e479' # a
        'e7c8376d8dd54ea96c56f4ea657aae08' # b
        'ba78252e1ca6b4c6e8dd741f4bbd8b8a' # c
        '703eb5664803f60e613557b986c11d9e' # d
        'e1f8981169d98e949b1e87e9ce5528df' # e
        '8ca1890dbfe6426841992d0fb054bb16' # f
    )

In [ ]:
def byte_substitution(state):
    return bytes(SBOX[byte] for byte in state)

def inverse_byte_substitution(state):
    return bytes(SBOX.index(byte) for byte in state)

In [ ]:
state_inp = randbytes(block_size)
state_enc = byte_substitution(state_inp)
state_dec = inverse_byte_substitution(state_enc)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6, 2))
show_state(state_inp, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')
show_state(state_dec, ax=ax3, title='decrypted')

plt.show()

### <span style="color:#2E8B57"><strong>Shift Rows</strong></span>:

The rows of the state are cyclically shifted over different numbers of bytes (offsets). Therefore, it simply consists in a bytes permutation of the state.

![Caption](images/shift_rows.png)

In [ ]:
def shift_rows(state: bytes) -> bytes:
    mat = np.array(list(state), dtype=np.ubyte).reshape(-1, 4).T
    mat = np.vstack([np.roll(row, -i) for i, row in enumerate(mat)])
    return bytes(byte for byte in mat.T.flatten())

def inverse_shift_rows(state: bytes) -> bytes:
    mat = np.array(list(state), dtype=np.ubyte).reshape(-1, 4).T
    mat = np.vstack([np.roll(row, i) for i, row in enumerate(mat)])
    return bytes(byte for byte in mat.T.flatten())


In [ ]:
state_inp = randbytes(block_size)
state_shifted = shift_rows(state_inp)
state_ = inverse_shift_rows(state_shifted)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6, 2))
show_state(state_inp, ax=ax1, title='input')
show_state(state_shifted, ax=ax2, title='shifted')
show_state(state_, ax=ax3, title='inversed')

plt.show()

### <span style="color:#2E8B57"><strong>Mix Columns</strong></span>:
It's the AES operation that gives diffusion: it mixes the bytes inside each column of the state, so that one input byte affects several output bytes.
It operates on the state column-by-column by computing a matrix-vector multiplication in which each element is in $GF(2^8)$,

When two bytes are multiplied as polynomials, the product may have degree greater than 7. Therefore, AES uses the following irreducible polynomial as fixed rule to reduce the product back into one byte. 

$$
P(x)=x^8+x^4+x^3+x+1.
$$



![Caption](images/mix_columns.png)

**Note** that there are **only three values to be multiplied** to a byte: 01, 02, and 03. Therefore, you do not need to implement the multiplication between two generic bytes but only the multiplication of a byte by those three values: **01∙𝐴**, **02∙𝐴**, and **03∙𝐴**.


**Implementation tricks**:

In MixColumns, bytes are only multiplied by the constants $01$, $02$, and $03$. Therefore, AES does not need generic multiplication between two arbitrary bytes in $GF(2^8)$.

Multiplying by $01$ leaves the byte unchanged:

$$
01 \cdot A = A.
$$

Multiplying by $02$ means multiplying the byte polynomial by $x$. In binary, this is almost the same as a left shift by one bit.

If the original most significant bit of $A$ is $0$, the left shift does not overflow, so:

$$
02 \cdot A = A \ll 1.
$$

If the original most significant bit of $A$ is $1$, the left shift produces a term of degree $8$, which does not fit in one byte. Therefore, AES reduces the result modulo

$$
P(x)=x^8+x^4+x^3+x+1.
$$

In practice, this reduction is implemented by XOR-ing with `0x1B`.

So the implementation rule is:

$$
02 \cdot A =
\begin{cases}
A \ll 1, & \text{if the MSB of } A \text{ is } 0,\\
(A \ll 1) \oplus \texttt{0x1B}, & \text{if the MSB of } A \text{ is } 1.
\end{cases}
$$

Multiplication by $03$ is computed using the fact that $03 = 02 \oplus 01$:

$$
03 \cdot A = (02 \cdot A) \oplus A.
$$

In [ ]:
def mul_by_x(byte: int) -> int:
    tmp = (int(byte) << 1) & 0xff
    if ((byte >> 7) & 0x01) > 0:
        tmp^= 0x1b  # computes mod P(x)
    return tmp

def gf_mul(byte: int, value: Literal[1, 2, 3]) -> int:
    """
    Multiplication used by AES MixColumns.
    Only supports multiplication by 01, 02, and 03.
    """
    if value == 0x01:
        return byte
    if value == 0x02:
        return mul_by_x(byte)
    if value == 0x03:
        return byte ^ mul_by_x(byte)
    raise ValueError("MixColumns only supports multiplication by 01, 02, and 03")


def matrix_product(A: NDArray[np.ubyte], X: NDArray[np.ubyte]) -> NDArray[np.ubyte]:
    """
    Matrix product over GF(2^8).
    """
    rows, inner = A.shape
    inner, cols = X.shape
    Y = np.zeros((rows, cols), dtype=np.ubyte)
    for i in range(rows):
        for j in range(cols):
            value = 0
            for k in range(inner):
                value ^= gf_mul(int(X[k, j]), A[i, k])
            Y[i, j] = value
    return Y


def mix_column(state: bytes):
    """
    Applies the AES MixColumns layer.
    """
    col = np.array([0x02, 0x01, 0x01, 0x03], dtype=np.ubyte)
    n = len(col)
    mat = circulant(col)
    X = np.array(list(state), dtype=np.ubyte).reshape(-1, n).T
    Y = matrix_product(mat, X)
    return bytes(byte for byte in Y.T.flatten())

In [ ]:
state_input = bytes.fromhex('6347a2f0f20a225c01010101c6c6c6c6d4d4d4d52d26314c')
state_enc = mix_column(state_input)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')

plt.show()

|  | Input | Encrypted |
|---:|---|---|
| 1 | `63 f2 01 c6 d4 2d` | `5d 9f 01 c6 d5 4d` | 
| 2 | `47 0a 01 c6 d4 26` | `e0 dc 01 c6 d5 7e` | 
| 3 | `a2 22 01 c6 d4 31` | `70 58 01 c6 d7 bd` | 
| 4 | `f0 5c 01 c6 d5 4c` | `bb 9d 01 c6 d6 f8` | 


### <span style="color:#2E8B57"><strong>Inverse Mix Columns</strong></span>:

In [ ]:
def mul_by_x(byte: int) -> int:
    tmp = (int(byte) << 1) & 0xff
    if ((byte >> 7) & 0x01) > 0:
        tmp^= 0x1b  # computes mod P(x)
    return tmp

def multiply(x, y):
    ''' Multiply-and-Sum Algorithm for multiplication '''
    z = 0
    for i in list(range(int(y).bit_length()))[::-1]:
        z = mul_by_x(z)
        if bool((y >> i) & 0x01):
            z ^= x
    return z

def dot_product(x, y):
    product = 0
    for xi, yi in zip(x, y):
        product^= multiply(xi, yi)
    return product

def matrix_product(X, Y):
    if len(X.shape) != 2 or len(Y.shape) != 2 or \
        X.shape[1] != Y.shape[0]:
        raise ValueError(f'Incompatible matrix shapes {X.shape} and {Y.shape}')
    (n, k), (k, m) = X.shape, Y.shape
    mat = np.zeros((n, m), np.ubyte)
    for i, row in enumerate(X):
        for j, col in enumerate(Y.T):
            mat[i, j] = dot_product(row, col)
    return mat

def _mix_column(state: bytes, col):
    n = len(col)
    mat = circulant(col)
    X = np.array(list(state), dtype=np.ubyte).reshape(-1, n).T
    Y = matrix_product(mat, X)
    return bytes(byte for byte in Y.T.flatten())

def mix_column(state: bytes) -> bytes:
    col = np.array([0x02, 0x01, 0x01, 0x03], dtype=np.ubyte)
    return _mix_column(state, col)

def inverse_mix_column(state: bytes) -> bytes:
    col = np.array([0x0e, 0x09, 0x0d, 0x0b], dtype=np.ubyte)
    return _mix_column(state, col)

In [ ]:
state_input = bytes.fromhex('6347a2f0f20a225c01010101c6c6c6c6d4d4d4d52d26314c')
state_enc = mix_column(state_input)
state_dec = inverse_mix_column(state_enc)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(9, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')
show_state(state_dec, ax=ax3, title='decrypted')

plt.show()

|  | Input | Encrypted | Decrypted |
|---:|---|---|---|
| 1 | `63 f2 01 c6 d4 2d` | `5d 9f 01 c6 d5 4d` | `63 f2 01 c6 d4 2d` |
| 2 | `47 0a 01 c6 d4 26` | `e0 dc 01 c6 d5 7e` | `47 0a 01 c6 d4 26` |
| 3 | `a2 22 01 c6 d4 31` | `70 58 01 c6 d7 bd` | `a2 22 01 c6 d4 31` |
| 4 | `f0 5c 01 c6 d5 4c` | `bb 9d 01 c6 d6 f8` | `f0 5c 01 c6 d5 4c` |


### Round

In [ ]:
def round(state: bytes , round_key: bytes) -> bytes:
    state = byte_substitution(state)
    state = shift_rows(state)
    state = mix_column(state)
    state = add_round_key(state, round_key)
    return state

def inverse_round(state: bytes , round_key: bytes) -> bytes:
    state = add_round_key(state, round_key)
    state = inverse_mix_column(state)
    state = inverse_shift_rows(state)
    state = inverse_byte_substitution(state)
    return state

In [ ]:
round_key = bytes.fromhex('a0fafe1788542cb123a339392a6c7605')
state_input = bytes.fromhex('193de3bea0f4e22b9ac68d2ae9f84808')
state_enc = round(state_input, round_key)
state_dec = inverse_round(state_enc, round_key)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')
show_state(state_dec, ax=ax3, title='decrypted')

plt.show()

| Row | Input | Encrypted | Decrypted |
|---:|---|---|---|
| 1 | `19 a0 9a e9` | `a4 68 6b 02` | `19 a0 9a e9` | 
| 2 | `3d f4 c6 f8` | `9c 9f 5b 6a` | `3d f4 c6 f8` | 
| 3 | `e3 e2 8d 48` | `7f 35 ea 50` | `e3 e2 8d 48` | 
| 4 | `be 2b 2a 08` | `f2 2b 43 49` | `be 2b 2a 08` | 

### Final Round

In [ ]:
def final_round(state: bytes , round_key: bytes) -> bytes:
    state = byte_substitution(state)
    state = shift_rows(state)
    state = add_round_key(state, round_key)
    return state

def inverse_final_round(state: bytes , round_key: bytes) -> bytes:
    state = add_round_key(state, round_key)
    state = inverse_shift_rows(state)
    state = inverse_byte_substitution(state)
    return state

In [ ]:
round_key = bytes.fromhex('d014f9a8c9ee2589e13f0cc8b6630ca6')
state_input = bytes.fromhex('eb40f21e592e38848ba113e71bc342d2')
state_enc = final_round(state_input, round_key)
state_dec = inverse_final_round(state_enc, round_key)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6, 2))
show_state(state_input, ax=ax1, title='input')
show_state(state_enc, ax=ax2, title='encrypted')
show_state(state_dec, ax=ax3, title='decrypted')

plt.show()

### <span style="color:grey"><strong>Key Expansion</strong></span>:

AES does not use the original cipher key directly in every round. Instead, it **expands the cipher key into several round keys**.

For AES-128, the cipher key has length $128$ bits, i.e. $16$ bytes or $4$ words. Therefore, $N_k=4$. Since AES-128 has $N_r=10$ rounds, the algorithm generates $N_r+1=11$ round keys:

$$
K_0, K_1, \dots, K_{10}.
$$

The first round key is the original cipher key:

$$
K_0 = K.
$$

Each round key contains $4$ words, i.e. $16$ bytes. The remaining round keys are generated word by word from the previous key material. For AES-128, the expansion is organized into $10$ stages, where each stage generates $N_k=4$ new words, corresponding to the next round key.


![Caption](images/key_expansion.png)

very stage implements a very similar transformation. The only difference is in the round coefficient (RC) that is different for each round.

![Caption](images/key_expansion_2.png)

In [ ]:
ROUND_CONSTS = (
    b'\x01\x00\x00\x00', b'\x02\x00\x00\x00',
    b'\x04\x00\x00\x00', b'\x08\x00\x00\x00',
    b'\x10\x00\x00\x00', b'\x20\x00\x00\x00',
    b'\x40\x00\x00\x00', b'\x80\x00\x00\x00',
    b'\x1b\x00\x00\x00', b'\x36\x00\x00\x00',
)


def substitution_word(word: bytes) -> bytes:
    return bytes([SBOX[byte] for byte in word])

def rotate_word(word: bytes) -> bytes:
    return bytes(list(word)[1:] + list(word[:1]))

def xor_words(x: bytes, y: bytes) -> bytes:
    return bytes([a ^ b for a, b in zip(x, y)])

def key_schedule(key) -> Iterator[bytes]:

    key_size = len(key)
    if key_size == 16:
        num_rounds = 10
    elif key_size == 24:
        num_rounds = 12
    else:
        num_rounds = 14

    num_words = key_size // 4
    total_words = 4 * (1 + num_rounds)

    words = []
    for i in range(num_words):
        words.append(bytes(byte for byte in key[4*i:4*(i+1)]))

    for j in range(num_words, total_words):
        word = words[j - 1]
        if j % num_words == 0:
            word = rotate_word(word)
            word = substitution_word(word)
            round_const = ROUND_CONSTS[j // num_words - 1]
            word = xor_words(word, round_const)
        elif (num_words == 8) and (j % 4 == 0):
            word = substitution_word(word)
        word = xor_words(word, words[j - num_words])
        words.append(word)

    for i in range(1 + num_rounds):
        yield b''.join(words[4*i:4*(i+1)])

def key_expansion(key) -> tuple[bytes, ...]:
    return tuple(key_schedule(key))

In [ ]:
def print_round_keys(round_keys):
    for i, round_key in enumerate(round_keys):
        print(f'{i:2d}', end=' ')
        for j, byte in enumerate(round_key):
            if j % 4 == 0:
                print('', end=' ')
            print(f'{byte:02x}', end='')
        print()

In [ ]:
key = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
round_keys = key_expansion(key)
print_round_keys(round_keys)


| Round | Word 0 | Word 1 | Word 2 | Word 3 |
|---:|---|---|---|---|
| 0 | `2b7e1516` | `28aed2a6` | `abf71588` | `09cf4f3c` |
| 1 | `a0fafe17` | `88542cb1` | `23a33939` | `2a6c7605` |
| 2 | `f2c295f2` | `7a96b943` | `5935807a` | `7359f67f` |
| 3 | `3d80477d` | `4716fe3e` | `1e237e44` | `6d7a883b` |
| 4 | `ef44a541` | `a8525b7f` | `b671253b` | `db0bad00` |
| 5 | `d4d1c6f8` | `7c839d87` | `caf2b8bc` | `11f915bc` |
| 6 | `6d88a37a` | `110b3efd` | `dbf98641` | `ca0093fd` |
| 7 | `4e54f70e` | `5f5fc9f3` | `84a64fb2` | `4ea6dc4f` |
| 8 | `ead27321` | `b58dbad2` | `312bf560` | `7f8d292f` |
| 9 | `ac7766f3` | `19fadc21` | `28d12941` | `575c006e` |
| 10 | `d014f9a8` | `c9ee2589` | `e13f0cc8` | `b6630ca6` |


### AES Chiper

In [ ]:
key = bytes.fromhex('00000000000000000000000000000000')
plaintext = bytes.fromhex('00000000000000000000000000000000')

aes = AES(key)
ciphertext = aes.encrypt_block(plaintext)
decrypted = aes.decrypt_block(ciphertext)

print(' plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print(' decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

|  |  |
|---|---|
| Plaintext | `00000000000000000000000000000000` |
| Ciphertext | `66e94bd4ef8a2c3b884cfa59ca342b2e` |
| Decrypted plaintext | `00000000000000000000000000000000` |

In [ ]:
key = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
plaintext = bytes.fromhex('3243f6a8885a308d313198a2e0370734')

aes = AES(key)
ciphertext = aes.encrypt_block(plaintext)
decrypted = aes.decrypt_block(ciphertext)

print(' plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print(' decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))


|  |  |
|---|---|
| Plaintext, hex | `3243f6a8885a308d313198a2e0370734` |
| Ciphertext, hex | `3925841d02dc09fbdc118597196a0b32` |
| Decrypted plaintext, hex | `3243f6a8885a308d313198a2e0370734` |

## Modes of Operation

Real-world messages are almost never exactly 16 bytes long. A natural first idea is to split the message into 16-byte blocks and encrypt each one independently with AES. This works, but it is **insecure**: AES is deterministic, so identical plaintext blocks always produce identical ciphertext blocks. An attacker who sees the ciphertext can detect repeated blocks and infer structure in the original message — without breaking AES at all.

**Modes of operation** are the solution. A mode defines a rule for how blocks are processed together, introducing dependencies between them so that the ciphertext of each block depends on all the previous ones. This way, even if the same 16-byte pattern appears twice in a message, it encrypts to a different ciphertext each time.

- **Electronic Codebook (ECB)**: the simplest encryption mode. The plaintext is divided into blocks, and each block is encrypted independently using the same key. Identical plaintext blocks produce identical ciphertext blocks, so ECB does not hide patterns well.

- **Cipher Block Chaining (CBC)**: each plaintext block is XORed with the previous ciphertext block before encryption. An initialization vector (**IV**) is used for the first block. This makes equal plaintext blocks encrypt differently depending on their position.

- **Output Feedback (OFB)**: the block cipher is used to generate a keystream, which is then XORed with the plaintext blocks to obtain the ciphertext. OFB turns the block cipher into a stream cipher mode.

- **Cipher Feedback (CFB)**: similar to OFB, it also generates a keystream that is XORed with the plaintext. However, CFB uses the previous ciphertext block to generate the next keystream block, instead of using the previous keystream block.

![Caption](images/modes_of_op.png)

#### Exercise

In [ ]:
def split_into_blocks(text: bytes, block_size: int) -> tuple[bytes, ...]:
    num_blocks = len(text) // block_size
    blocks = []
    for i in range(num_blocks):
        blocks.append(text[i*block_size:(i+1)*block_size])
    return tuple(blocks)

def xor(x: bytes, y: bytes) -> bytes:
    return bytes([xi ^ yi for xi, yi in zip(x, y)])

def encrypt_ecb(aes: AES, plaintext: bytes) -> bytes:
    ciphertext = b''
    for block in split_into_blocks(plaintext, aes.block_size):
        ciphertext += aes.encrypt_block(block)
    return ciphertext

def decrypt_ecb(aes: AES, ciphertext: bytes) -> bytes:
    plaintext = b''
    for block in split_into_blocks(ciphertext, aes.block_size):
        plaintext += aes.decrypt_block(block)
    return plaintext

def encrypt_cbc(aes: AES, plaintext: bytes, iv: bytes) -> bytes:
    ciphertext = b''
    prev = iv
    for block in split_into_blocks(plaintext, aes.block_size):
        prev = aes.encrypt_block(xor(prev, block))
        ciphertext += prev
    return ciphertext

def decrypt_cbc(aes: AES, ciphertext: bytes, iv: bytes) -> bytes:
    plaintext = b''
    prev = iv
    for block in split_into_blocks(ciphertext, aes.block_size):
        plaintext += xor(prev, aes.decrypt_block(block))
        prev = block
    return plaintext

def crypt_ofb(aes: AES, text: bytes, iv: bytes) -> bytes:
    output_text = b''
    prev = iv
    for block in split_into_blocks(text, aes.block_size):
        prev = aes.encrypt_block(prev)
        output_text += xor(prev, block)
    return output_text

encrypt_ofb = crypt_ofb
decrypt_ofb = crypt_ofb

def encrypt_cfb(aes: AES, plaintext: bytes, iv: bytes) -> bytes:
    ciphertext = b''
    prev = iv
    for block in split_into_blocks(plaintext, aes.block_size):
        tmp = aes.encrypt_block(prev)
        prev = xor(block, tmp)
        ciphertext += prev
    return ciphertext

def decrypt_cfb(aes: AES, ciphertext: bytes, iv: bytes) -> bytes:
    plaintext = b''
    prev = iv
    for block in split_into_blocks(ciphertext, aes.block_size):
        tmp = aes.encrypt_block(prev)
        plaintext += xor(block, tmp)
        prev = block
    return plaintext

In [ ]:
key = bytes.fromhex('00000000000000000000000000000000')
iv = bytes.fromhex('00000000000000000000000000000000')
plaintext = bytes.fromhex('00000000000000000000000000000000')
print('     plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('            iv:', ''.join([f'{byte:02x}' for byte in plaintext]))

aes = AES(key)

ciphertext = encrypt_ecb(aes, plaintext)
decrypted = decrypt_ecb(aes, ciphertext)
print('ECB ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print('ECB  decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

ciphertext = encrypt_cbc(aes, plaintext, iv)
decrypted = decrypt_cbc(aes, ciphertext, iv)
print('CBC ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print('CBC  decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

ciphertext = encrypt_ofb(aes, plaintext, iv)
decrypted = decrypt_ofb(aes, ciphertext, iv)
print('OFB ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print('OFB  decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

ciphertext = encrypt_cfb(aes, plaintext, iv)
decrypted = decrypt_cfb(aes, ciphertext, iv)
print('CFB ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print('CFB  decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

### Padding with PKCS7

Block Ciphers encrypt data in fixed-size blocks (e.g., 128 bits = 16 bytes in AES). What if plaintext is not multiple of the block size?

Padding ensures plaintext is a multiple of the block size.
- Prevent data truncation
- Ensure deterministic block alignment for encryption

Common padding schemes: 
- [PKCS7](https://datatracker.ietf.org/doc/html/rfc5652)
- [ANSI X.923](https://www.ibm.com/docs/en/linux-on-systems?topic=processes-ansi-x923-cipher-block-chaining)
- [ISO/IEC 7816-4](https://www.iso.org/standard/77180.html) (OneAndZeroes)


The rules for PKCS padding are:
- **Padding bytes are always added** to the clear text before it is encrypted.
- Each **padding byte has a value equal to the total number of padding bytes** that are added. 
- The total number of padding bytes is at least one, and is the minimum number required to bring the data length up to a multiple of the block size.

Example 1:

|            | bytes                                             |
|:-----------|:--------------------------------------------------|
| message 01 | `39 25 84 1d 02 dc 09 fb dc 11 85 97 19 6a`       |
| padded  01 | `39 25 84 1d 02 dc 09 fb dc 11 85 97 19 6a 02 02` |

Example 2:

|            | bytes                                             |
|:-----------|:--------------------------------------------------|
| message 01 | `39 25 84 1d 02 dc 09 fb dc 11 85 97 19 6a 0b 32` |
| message 02 | `32 43 f6 a8 88 5a`                               |
| padded  01 | `39 25 84 1d 02 dc 09 fb dc 11 85 97 19 6a 0b 32` |
| padded  02 | `32 43 f6 a8 88 5a 0a 0a 0a 0a 0a 0a 0a 0a 0a 0a` |

Example 3:

|            | bytes                                             |
|:-----------|:--------------------------------------------------|
| message 01 | `39 25 84 1d 02 dc 09 fb dc 11 85 97 19 6a 0b 32` |
| padded  01 | `39 25 84 1d 02 dc 09 fb dc 11 85 97 19 6a 0b 32` |
| padded  02 | `10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10` |


#### Exercise

define a `pad` and a `unpad` functions that implement the PKCS standard

In [ ]:
def pad(text: bytes, block_size: int) -> bytes:
    length = block_size - len(text) % block_size
    padding = bytes([length] * length)
    return text + padding

def unpad(padded_text: bytes, block_size: int) -> bytes:
    if len(padded_text) == 0 or len(padded_text) % block_size != 0:
        raise ValueError()
    length = padded_text[-1]
    if length == 0 or length > block_size \
        or padded_text[-length:] != bytes([length] * length):
        raise ValueError()
    return padded_text[:-length]

In [ ]:
def print_bytes(text: bytes, sep: str = ''):
    print(sep.join([f'{byte:02x}' for byte in text]))

block_size = 16
text_list = [
    bytes.fromhex('3925841d02dc09fbdc118597196a'),
    bytes.fromhex('3925841d02dc09fbdc118597196a0b323243f6a8885a'),
    bytes.fromhex('3925841d02dc09fbdc118597196a0b32'),
]

for text in text_list:
    padded_text = pad(text, block_size)
    unpadded_text = unpad(padded_text, block_size)

    print('    text:', ''.join([f'{byte:02x}' for byte in text]))
    print('  padded:', ''.join([f'{byte:02x}' for byte in padded_text]))
    print('unpadded:', ''.join([f'{byte:02x}' for byte in unpadded_text]))
    print()

#### Exercise

Create a `BlockCipher` base class that implements **ECB, CBC, OFB, CFB** modes of operation and deals with **padding**.
This class is meant to be a base class for `AES` class, so that `AES` can be defined as a child class of `BlockCipher` and inherit all methods related to modes of operation.

In [ ]:
# see implementation of BlockCipher in blockcipher.py and AES in aes.py

In [ ]:
key = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
plaintext = bytes.fromhex('3243f6a8885a308d313198a2e0370734')

aes = AES(key)
ciphertext = aes.encrypt_block(plaintext)
decrypted = aes.decrypt_block(ciphertext)

print(' plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print(' decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

In [ ]:
key = bytes.fromhex('2b7e151628aed2a6abf7158809cf4f3c')
plaintext = bytes.fromhex('3243f6a8885a308d313198a2e0370734')

aes = AES(key)
ciphertext = aes.encrypt(plaintext)
decrypted = aes.decrypt(ciphertext)

print(' plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))
print(' decrypted:', ''.join([f'{byte:02x}' for byte in decrypted]))

#### Excercise

encrypt an image with all implemented modes of operation and observe the differences

In [ ]:
image_original = imread('images/penguin.png')  # load image

# make image black and white
image = (255 * (image_original[...,:3].mean(axis=-1) > 0.5)).astype(np.ubyte)

fig, axs = plt.subplots(1, 2)
axs[0].imshow(image_original, cmap='gray')
axs[1].imshow(image, cmap='gray')
for ax in axs:
    ax.axis('off')

In [ ]:
# plaintext obtained by flattening the image and convert into bytes
plaintext = bytes(image.flatten())

# to convert bytes to image, need a proper function
def bytes_to_image(text: bytes) -> NDArray[np.ubyte]:
    ciphertext_image = np.array(
        list(text[:image.size]),
        dtype=np.ubyte
    ).reshape(*image.shape)
    return ciphertext_image


In [ ]:
plaintext = bytes(image.flatten())
key = b'0123456701234567'
iv = b'abcdefghijklmnop'

ciphertext_images = {}
for mode in BlockCipher.modes:
    aes = AES(key, iv=iv, mode=mode)
    ciphertext = aes.encrypt(plaintext)
    ciphertext_images[mode] = bytes_to_image(ciphertext)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
for ax, (mode, ciphertext_image) in zip(axs.flatten(), ciphertext_images.items()):
    ax.imshow(ciphertext_image, cmap='gray')
    ax.set(title=mode)
    ax.axis('off')

## Diffusion and Confusion in AES

**Diffusion:** if we change a single bit of the plaintext, then statistically half of the bits in the ciphertext should change. 

To **test** if AES provides diffusion, we randomly draw a plaintext $x$ and a key $k$, change a bit in the plaintext, and then observe how this change affects the ciphertext.


![Caption](images/diffusion.png)

**Confusion**: the relationship between the key and the ciphertext must be very complicated and impossible (hard) to invert. 

At bit level, it means that **each bit of the ciphertext** **depends** on **several parts of the key**. As a result, **if a single bit** is changed in the **key**, **many bits** in the **ciphertext** change.

To **test** it we consider two keys 𝑘 and 𝑘′ that differ only by **1bit**. What is the **distribution of the Hamming distance** between the ciphertext **𝑦** and **𝑦′** obtained by encrypting  the **same plaintext** 𝑥 with the two **different keys** 𝑘 and 𝑘′?


![Caption](images/confusion.png)

### Hamming Distance

We measure the difference between $y$ and $y'$ using the **Hamming distance**. For two bit strings $a = (a_1, a_2, \ldots, a_n)$, where $a_i \in \{0,1\}$, and  $b = (b_1, b_2, \ldots, b_n)$, where $b_i \in \{0,1\}$, the Hamming distance is

$$
d(a,b) = \sum_{i=1}^{n} |a_i - b_i|.
$$

Example: $a = (1,0,1,1,0,0)$, $b = (1,1,0,1,1,0)$, then $d(a,b) = 3$

In [ ]:
def hamming_distance(x, y):
    dist = sum((a ^ b).bit_count() for a, b in zip_longest(x, y, fillvalue=0))
    return dist

|  |  |
|---|---|
| $x$ | `10000111 01111000 00010100 01100110` |
| $y$ | `00001111 11101111 11000000 10111110` |
| $x \oplus y$ | `10001000 10010111 11010100 11011000` |
| $d(x,y)$ | `15` |

In [ ]:
length = 4
x = randbytes(length)
y = randbytes(length)

distance = hamming_distance(x, y)

print( '     x:', ' '.join(f'{byte:08b}' for byte in x))
print( '     y:', ' '.join(f'{byte:08b}' for byte in y))
print( ' x ^ y:', ' '.join(f'{a ^ b:08b}' for a, b in zip(x, y)))

print(f'd(x,y): {distance}')

### Monte Carlo Simulation

Rather than considering a single value of the Hamming distance, we are interested in estimating the **distribution** of this distance.

This distribution can be approximated using **Monte Carlo simulations**, a statistical technique used to model complex systems involving random variables. Monte Carlo methods repeatedly perform random sampling in order to study stochastic systems and estimate the probabilities of different outcomes.


**Observation**: Let $ d(y, y') $ denote the Hamming distance between two ciphertexts $y$ and $y'$. We do not expect $d(y, y') = \frac{n}{2}$ to hold for every pair of ciphertexts, where $n$ is the number of bits in the ciphertext. Instead, we expect the values of $d(y, y')$ to follow a **probability distribution that is concentrated around $ \frac{n}{2}.$**

![Caption](images/pdf.png)

#### MonteCarlo functions

Analyze the **diffusion** and **confusion** properties of the AES encryption function by estimating the distribution of the **Hamming distance** between two ciphertexts generated from **plaintexts** **or** **keys** that differ by exactly one bit.

- Estimate the distribution using **Monte Carlo simulation**.
- Perform the analysis **between AES rounds** to observe how diffusion and confusion evolve throughout the encryption data path.
- Discuss and interpret the obtained results.

`Partially_encrypt`: a method to be added to `AES` that encrypts a block using the number of rounds indicated as an argument.

In [ ]:
def partially_encrypt(
        self: AES,
        plaintext: bytes,
        num_rounds: int | None = None
        ) -> bytes:
    if num_rounds is None:
        num_rounds = self.num_rounds
    num_rounds = min(num_rounds, self.num_rounds)
    round_keys = list(self.key_schedule())[:num_rounds + 1]
    state = self.add_round_key(plaintext, round_keys[0])
    if num_rounds > 0:
        for round_key in round_keys[1:min(self.num_rounds, num_rounds + 1)]:
            state = self.round(state, round_key)
        if num_rounds == self.num_rounds:
            state = self.final_round(state, round_keys[-1])
    return state

AES.partially_encrypt = partially_encrypt

In [ ]:
key_size = AES.KEY_SIZES[0]
key = bytes(key_size)  # 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
plaintext = bytes(key_size)
print('      key:', ''.join([f'{byte:02x}' for byte in key]))
print('plaintext:', ''.join([f'{byte:02x}' for byte in plaintext]))

aes = AES(key)
for num_rounds in range(aes.num_rounds + 1):
    partial = aes.partially_encrypt(plaintext, num_rounds)
    print(f'  round {num_rounds:2d}:',
          ''.join([f'{byte:02x}' for byte in partial]))
ciphertext = aes.encrypt_block(plaintext)
print('ciphertext:', ''.join([f'{byte:02x}' for byte in ciphertext]))

```text
       key: 00000000000000000000000000000000
 plaintext: 00000000000000000000000000000000
  round  0: 00000000000000000000000000000000
  round  1: 01000000010000000100000001000000
  round  2: c6e4e48ba48787e8c6e4e48ba48787e8
  round  3: 282df3c46af386254a4e90a70890e546
  round  4: abd2cdfe375ab54950a0afc0759a6a5f
  round  5: d46f4f6c55b896337e05bb3d7979de23
  round  6: 04f2ca9707782845e22f019649c5d710
  round  7: b7aae4c51d252d4f6c920f8194e58150
  round  8: 23e78c3c132163dbaac0c6572e03cb95
  round  9: 7ffe0e9551a566350e347c472929eccb
  round 10: 66e94bd4ef8a2c3b884cfa59ca342b2e
ciphertext: 66e94bd4ef8a2c3b884cfa59ca342b2e
```

`flip_bit`: flips one bit in a bytes object

In [ ]:
def flip_bit(text, index=None):
    if index is None:
        index = randrange(8 * len(text))

    i_byte, i_bit = divmod(index, 8)
    flipped = bytearray(text)
    flipped[i_byte] ^= 1 << (7 - i_bit)
    return bytes(flipped)

In [ ]:
state = b'\xa5'
states = [state for _ in range(8)]
flipped = [flip_bit(state, i) for i in range(8)]

print('  input:', ' '.join(f'{state[0]:08b}' for state in states))
print('flipped:', ' '.join(f'{state[0]:08b}' for state in flipped))
print('        ',' '.join(' '*i + '^' + ' '*(7-i) for i in range(8)))

```text
10100101 10100101 10100101 10100101 10100101 10100101 10100101 10100101
00100101 11100101 10000101 10110101 10101101 10100001 10100111 10100100
^        ^        ^        ^        ^        ^        ^        ^

`aes_diffusion`: generates a random plaintext and a random key, flips one bit of the plaintext, and returns the Hamming distance between the two resulting ciphertexts.

In [ ]:
def aes_diffusion(key_size=16, num_rounds=None):

   key = randbytes(key_size)
   aes = AES(key)

   x1 = randbytes(AES.block_size)
   y1 = aes.partially_encrypt(x1, num_rounds)

   index = randrange(8 * AES.block_size)
   x2 = flip_bit(x1, index)
   y2 = aes.partially_encrypt(x2, num_rounds)

   return hamming_distance(y1, y2)

In [ ]:
distance = aes_diffusion()
print(f'hamming distance: {distance}')

`aes_confusion`: generates a random plaintext and key, flips a bit of the key and returns the hamming distance between the two corresponding ciphertexts.

In [ ]:
def aes_confusion(key_size=16, num_rounds=None):

    x = randbytes(AES.block_size)

    k1 = randbytes(key_size)
    y1 = AES(k1).partially_encrypt(x, num_rounds=num_rounds)

    index = randrange(8 * key_size)
    k2 = flip_bit(k1, index)
    y2 = AES(k2).partially_encrypt(x, num_rounds=num_rounds)

    distance = hamming_distance(y1, y2)

    return distance

In [ ]:
distance = aes_confusion()
print(f'hamming distance: {distance}')

#### Simulations

In [ ]:
# it may take a while, be patient or reduce the number of trials

num_trials = 100  # 1000, 10_000
num_rounds_max = 10
dist_diff = np.empty((num_trials, num_rounds_max), dtype=int)
dist_conf = np.empty((num_trials, num_rounds_max), dtype=int)

for i, j in product(range(num_trials), range(num_rounds_max)):
    dist_diff[i, j] = aes_diffusion(num_rounds=j + 1)
    dist_conf[i, j] = aes_confusion(num_rounds=j + 1)

In [ ]:
ebins = np.arange(0, 8 * AES.block_size + 1) - 0.5

tmp = np.max(np.abs(dist_diff - 4*AES.block_size))
xlim = (4*AES.block_size - tmp, 4*AES.block_size + tmp)

fig, ax = plt.subplots()
ax.set(title='diffusion')
for j, col in enumerate(dist_diff.T):
    ax.hist(col, bins=ebins, density=True, label=f'{j + 1} rounds', alpha=0.5)
ax.set(xticks=8*np.arange(0, AES.block_size), xlim=xlim)
ax.set(ylim=(0., 0.135))
ax.set(xlabel='hamming distance', ylabel='prob. density')
ax.legend()
ax.grid()

plt.show()

In [ ]:
rounds = np.arange(dist_diff.shape[-1]) + 1
_mean = np.mean(dist_diff, axis=0)
_std = np.std(dist_diff, axis=0)
_min = np.min(dist_diff, axis=0)
_max = np.max(dist_diff, axis=0)

fig, ax = plt.subplots()
ax.plot(rounds, _mean, label='mean', marker='o')
ax.fill_between(rounds, _mean - _std, _mean + _std, alpha=0.2, label='std')
ax.fill_between(rounds, _min, _max, alpha=0.1, color='C0', label='min-max')
ax.set(xticks=rounds, yticks=np.arange(0, max(_max), 16), ylim=(0, 94))
ax.set(xlabel='AES round', ylabel='Hamming distance', title='diffusion')
ax.legend()
ax.grid(True, ls=':')
plt.show()

In [ ]:
key_size = 16
ebins = np.arange(0, 8 * key_size + 1) - 0.5

tmp = np.max(np.abs(dist_conf - 4*key_size))
xlim = (4*key_size - tmp, 4*key_size + tmp)

fig, ax = plt.subplots()
ax.set(title='confusion')
for j, col in enumerate(dist_conf.T):
    ax.hist(col, bins=ebins, density=True, label=f'{j + 1} rounds', alpha=0.5)
ax.set(xticks=np.arange(0, 8*key_size, 8), xlim=xlim)
ax.set(xlabel='hamming distance', ylabel='prob. density')
ax.set(ylim=(0., 0.135))
ax.legend()
ax.grid()

plt.show()

In [ ]:
rounds = np.arange(dist_conf.shape[-1]) + 1
_mean = np.mean(dist_conf, axis=0)
_std = np.std(dist_conf, axis=0)
_min = np.min(dist_conf, axis=0)
_max = np.max(dist_conf, axis=0)

fig, ax = plt.subplots()
ax.plot(rounds, _mean, label='mean', marker='o')
ax.fill_between(rounds, _mean - _std, _mean + _std, alpha=0.2, label='std')
ax.fill_between(rounds, _min, _max, alpha=0.1, color='C0', label='min-max')
ax.set(xticks=rounds, yticks=np.arange(0, max(_max), 16), ylim=(0, 94))
ax.set(xlabel='AES round', ylabel='Hamming distance', title='confusion')
ax.legend()
ax.grid(True, ls=':')
plt.show()